# Imports

In [1]:
from WS_Mdl.core import *

In [2]:
import datetime as DT
import pandas as pd

# Prep

In [3]:
MdlN = 'NBr55'

In [4]:
M = Mdl_N(MdlN)

Could not determine imod version from Sim/NBr55 folder.
Proceeding with the assumption that it's imod_python.


In [5]:
MB = Mdl_N(M.B)

In [6]:
Pa_SFR_Out = M.Pa_B.Sim_In / f'{M.B}.SFR6.obs.output.csv'
DF = pd.read_csv(Pa_SFR_Out)

In [7]:
DF['date'] = pd.to_datetime(DF['time'], unit='d', origin=pd.Timestamp(MB.INI.SDATE)) - DT.timedelta(days=1) # Convert time to date

In [8]:
DF['month'] = DF['date'].dt.month

In [9]:
DF = DF[['date', 'month'] + [i for i in DF.columns if 'STG' in i]] # Only keep stage columns

In [10]:
DF.loc['AVG_Stg_summer'] = DF.loc[(DF['month']>3) & (DF['month']<10)].mean(axis='index')
DF.loc['AVG_Stg_winter'] = DF.loc[(DF['month']>=10) | (DF['month']<=3)].mean(axis='index')

In [11]:
DF.drop([i for i in DF.index if i not in ['AVG_Stg_summer', 'AVG_Stg_winter']], inplace=True)

In [12]:
DF = DF.drop(['date', 'month'], axis='columns').T

In [13]:
DF

,AVG_Stg_summer,AVG_Stg_winter
STG_L7_R282_C393,18.800000,18.800000
STG_L7_R281_C393,18.800000,18.800000
STG_L7_R281_C392,18.800000,18.800000
STG_L7_R281_C391,18.800000,18.800000
STG_L7_R281_C390,18.800000,18.800000
...,...,...
STG_L5_R24_C17,0.101187,0.153291
STG_L5_R24_C16,0.101461,0.153565
STG_L5_R23_C16,-0.075821,-0.069574
STG_L5_R23_C15,-0.600974,-0.549633


In [14]:
DF[['L', 'R', 'C']] = DF.index.to_series().str.extract(r'L(\d+)_R(\d+)_C(\d+)').astype(int)

In [15]:
DF = DF.ws.Calc_XY(MB.Xmin, MB.Ymax, MB.cellsize)

In [16]:
DF

,AVG_Stg_summer,AVG_Stg_winter,L,R,C,x,y
STG_L7_R282_C393,18.800000,18.800000,7,282,393,122912.5,389162.5
STG_L7_R281_C393,18.800000,18.800000,7,281,393,122912.5,389187.5
STG_L7_R281_C392,18.800000,18.800000,7,281,392,122887.5,389187.5
STG_L7_R281_C391,18.800000,18.800000,7,281,391,122862.5,389187.5
STG_L7_R281_C390,18.800000,18.800000,7,281,390,122837.5,389187.5
...,...,...,...,...,...,...,...
STG_L5_R24_C17,0.101187,0.153291,5,24,17,113512.5,395612.5
STG_L5_R24_C16,0.101461,0.153565,5,24,16,113487.5,395612.5
STG_L5_R23_C16,-0.075821,-0.069574,5,23,16,113487.5,395637.5
STG_L5_R23_C15,-0.600974,-0.549633,5,23,15,113462.5,395637.5


In [17]:
DA = DF.set_index(['y', 'x'])[['AVG_Stg_summer', 'AVG_Stg_winter']].to_xarray()

In [ ]:
import imod

In [22]:
Pa_Out = MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_AVG_Stg_summer_{MB.MdlN}.IDF'
imod.idf.save(Pa_Out, DA['AVG_Stg_summer'])

In [21]:
Pa_Out = MB.Pa.PoP_Out_MdlN / f'SFR/Stg/SFR_AVG_Stg_winter_{MB.MdlN}.IDF'
imod.idf.save(Pa_Out, DA['AVG_Stg_winter'])